<a href="https://colab.research.google.com/github/mmilannaik/bostonhousepricing/blob/main/W16S1_SQL_Window_01Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install pandasql
!pip install pandasql --quiet

# 2. Load libraries and your CSV into a pandas DataFrame
import pandas as pd
from pandasql import sqldf

# 3. Create a helper to run SQL against any DataFrame in your notebook
pysqldf = lambda query: sqldf(query, globals())

  Preparing metadata (setup.py) ... done


**Note: Try to avoid *GROUP BY* clause to solve the problems**

For the problems use the *Health Insurance Claim* dataset. You can get the details as well as the dataset from [here](https://www.kaggle.com/datasets/thedevastator/insurance-claim-analysis-demographic-and-health).

### **Problem 1:** What are the top 5 patients who claimed the highest insurance amounts?

### **Problem 2:** What is the average insurance claimed by patients based on the number of children they have?

### **Problem 3:** What is the highest and lowest claimed amount by patients in each region?

### **Problem 4:** What is the percentage of smokers in each age group?

### **Problem 5:** What is the difference between the claimed amount of each patient and the first claimed amount of that patient?

### **Problem 6:** For each patient, calculate the difference between their claimed amount and the average claimed amount of patients with the same number of children.

### **Problem 7:** Show the patient with the highest BMI in each region and their respective rank.

### **Problem 8:** Calculate the difference between the claimed amount of each patient and the claimed amount of the patient who has the highest BMI in their region.

### **Problem 9:** For each patient, calculate the difference in claim amount between the patient and the patient with the highest claim amount among patients with the same bmi and smoker status, within the same region. Return the result in descending order difference.

### **Problem 10:** For each patient, find the maximum BMI value among their next three records (ordered by age).

### **Problem 11:** For each patient, find the rolling average of the last 2 claims.

### **Problem 12:** Find the first claimed insurance value for male and female patients, within each region order the data by patient age in ascending order, and only include patients who are non-diabetic and have a bmi value between 25 and 30.

In [2]:
insur = pd.read_csv('/content/W16S1_Task_insurance_data.csv')
insur.head(2)

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim
0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87
1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51


In [3]:
insur['children'].value_counts()

,count
children,
0,576
1,324
2,240
3,157
4,25
5,18


In [10]:
insur['age'].value_counts().sort_index().head()

,count
age,
18.0,16
19.0,29
20.0,26
21.0,18
22.0,24


# Q1 :Problem 1: What are the top 5 patients who claimed the highest insurance amounts?

In [4]:
pysqldf('''
Select *  FROM (SELECT PatientID,
RANK() OVER(ORDER BY SUM(claim) DESC) as 'rnk'
FROM insur
GROUP BY PatientID ) t
WHERE rnk<=5



''')

,PatientID,rnk
0,1340,1
1,1339,2
2,1338,3
3,1337,4
4,1336,5


# Q2 Problem 2: What is the average insurance claimed by patients based on the number of children they have?

In [5]:
pysqldf('''
SELECT children,avg_claim FROM
(
SELECT *,
AVG(claim) OVER(PARTITION BY children) AS 'avg_claim',
ROW_NUMBER() OVER(PARTITION BY children) as 'row_no'
FROM insur
) t
WHERE t.row_no =1
''')

,children,avg_claim
0,0,12327.993160
1,1,12731.171821
2,2,15073.564000
3,3,15355.318535
4,4,13850.656800
5,5,8786.035556


# Q3 : Problem 3: What is the highest and lowest claimed amount by patients in each region?

In [6]:
pysqldf('''
SELECT region,claim
FROM
(
SELECT *,
RANK() OVER(PARTITION BY region ORDER BY claim DESC) AS 'rank_region_DESC',
RANK() OVER(PARTITION BY region ORDER BY claim ASC) AS 'rank_region_ASC'

FROM insur
) t
WHERE t.rank_region_DESC = 1 OR t.rank_region_ASC =1
''')

,region,claim
0,None,1256.30
1,None,1252.41
2,northeast,58571.07
3,northeast,1694.80
4,northwest,60021.40
5,northwest,1136.40
6,southeast,63770.43
7,southeast,1121.87
8,southwest,52590.83
9,southwest,1261.44


# Q4 Problem 4: What is the percentage of smokers in each age group?

In [20]:
pysqldf('''

SELECT age,
  COUNT(*) OVER(PARTITION BY smoker) as 'smoker_stats'
  FROM insur
  WHERE smoker = 'No'
  '''
)

,age,smoker_stats
0,39.0,1066
1,24.0,1066
2,NaN,1066
3,NaN,1066
4,NaN,1066
...,...,...
1061,18.0,1066
1062,29.0,1066
1063,32.0,1066
1064,55.0,1066


In [16]:
pysqldf('''
SELECT t.age,t.smoker_stats,
COUNT(*) OVER(PARTITION BY smoker) as 'full_stats'
FROM  (
  SELECT *,
  COUNT(*) OVER(PARTITION BY smoker) as 'smoker_stats'
  FROM insur
  WHERE smoker = 'Yes'

) t



''')

,age,smoker_stats,full_stats
0,38.0,274,274
1,33.0,274,274
2,52.0,274,274
3,48.0,274,274
4,52.0,274,274
...,...,...,...
269,44.0,274,274
270,59.0,274,274
271,30.0,274,274
272,37.0,274,274


In [21]:
insur.head(2)

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim
0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87
1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51


# Q5:What is the difference between the claimed amount of each patient and the first claimed amount of that patient?

In [31]:
pysqldf(
    """
    SELECT
        PatientID,
        claim,
        FIRST_VALUE(claim) OVER(PARTITION BY PatientID ORDER BY "index") AS first_claim,
        claim - FIRST_VALUE(claim) OVER(PARTITION BY PatientID ORDER BY "index") AS diff_claim
    FROM insur
    """
)


,PatientID,claim,first_claim,diff_claim
0,1,1121.87,1121.87,0.0
1,2,1131.51,1131.51,0.0
2,3,1135.94,1135.94,0.0
3,4,1136.40,1136.40,0.0
4,5,1137.01,1137.01,0.0
...,...,...,...,...
1335,1336,55135.40,55135.40,0.0
1336,1337,58571.07,58571.07,0.0
1337,1338,60021.40,60021.40,0.0
1338,1339,62592.87,62592.87,0.0


# Q6:For each patient, calculate the difference between their claimed amount and the average claimed amount of patients with the same number of children.

In [32]:
pysqldf(
    '''
    SELECT *,
    claim-AVG(claim) OVER(PARTITION BY children)
    FROm insur

    '''

)

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,claim-AVG(claim) OVER(PARTITION BY children)
0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87,-11206.123160
1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51,-11196.483160
2,2,3,NaN,male,33.3,82,Yes,0,No,southeast,1135.94,-11192.053160
3,3,4,NaN,male,33.7,80,No,0,No,northwest,1136.40,-11191.593160
4,4,5,NaN,male,34.1,100,No,0,No,northwest,1137.01,-11190.983160
...,...,...,...,...,...,...,...,...,...,...,...,...
1335,718,719,34.0,male,25.8,100,Yes,5,No,southwest,10096.97,1310.934444
1336,815,816,29.0,female,31.9,90,Yes,5,No,southwest,11552.90,2766.864444
1337,880,881,28.0,female,46.8,106,No,5,No,southeast,12592.53,3806.494444
1338,975,976,50.0,male,25.5,84,No,5,No,southeast,14478.33,5692.294444


# Q7 :Problem 7: Show the patient with the highest BMI in each region and their respective rank.

In [40]:
pysqldf('''

SELECT * FROM
(
SELECT *,
RANK() OVER(PARTITION BY region ORDER BY bmi desc) AS 'rank_region'
FROM insur
)t
WHERE t.rank_region =1
''')

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,rank_region
0,15,16,32.0,male,30.4,86,Yes,0,No,None,1256.30,1
1,675,676,49.0,female,48.1,81,Yes,2,No,northeast,9432.93,1
2,9,10,30.0,male,53.1,97,No,0,No,northwest,1163.46,1
3,1299,1300,50.0,male,52.6,110,No,1,Yes,southeast,44501.40,1
4,1306,1307,43.0,female,47.6,112,Yes,2,Yes,southwest,46113.51,1


In [37]:
insur.head(2)

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim
0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87
1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51


# Q8:Problem 8: Calculate the difference between the claimed amount of each patient and the claimed amount of the patient who has the highest BMI in their region.

In [46]:
pysqldf(
    '''
SELECT PatientID,
claim,
FIRST_VALUE(claim) OVER(PARTITION BY region ORDER BY bmi desc) AS 'high_bmi_claim'
FROM insur

'''

)

,PatientID,claim,high_bmi_claim
0,16,1256.30,1256.30
1,15,1253.94,1256.30
2,14,1252.41,1256.30
3,676,9432.93,9432.93
4,682,9541.70,9432.93
...,...,...,...
1335,63,1728.90,46113.51
1336,343,4766.02,46113.51
1337,1047,19023.26,46113.51
1338,62,1727.79,46113.51


# Q9:Problem 9: For each patient, calculate the difference in claim amount between the patient and the patient with the highest claim amount among patients with the same bmi and smoker status, within the same region. Return the result in descending order difference.

In [49]:
pysqldf('''
SELECT *,
(MAX(claim) OVER(PARTITION BY region,smoker)-claim) AS 'claim_diff'
FROM insur
ORDER BY claim_diff DESC

''')

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,claim_diff
0,1003,1004,42.0,female,24.8,83,No,0,Yes,southeast,16577.78,47192.65
1,1009,1010,22.0,male,27.1,86,Yes,0,Yes,southeast,17043.34,46727.09
2,1011,1012,32.0,female,26.8,109,No,1,Yes,southeast,17085.27,46685.16
3,1014,1015,44.0,male,19.8,106,No,1,Yes,southeast,17179.52,46590.91
4,1021,1022,19.0,male,24.0,80,No,3,Yes,southeast,17663.14,46107.29
...,...,...,...,...,...,...,...,...,...,...,...,...
1335,1337,1338,30.0,male,34.5,91,Yes,3,Yes,northwest,60021.40,0.00
1336,1222,1223,55.0,female,33.3,86,No,4,No,southeast,36580.28,0.00
1337,1339,1340,30.0,female,47.4,101,No,0,Yes,southeast,63770.43,0.00
1338,1225,1226,50.0,female,34.8,140,Yes,2,No,southwest,36910.61,0.00


# Q10:Problem 10: For each patient, find the maximum BMI value among their next three records (ordered by age).

In [52]:
pysqldf(
    '''
    SELECT PatientID,
    MAX(bmi) OVER(ORDER BY AGE ROWS BETWEEN UNBOUNDED PRECEDING AND 3 FOLLOWING) AS'max_bmi_unbounded_p3',

    MAX(bmi) OVER(ORDER BY AGE ROWS BETWEEN UNBOUNDED PRECEDING AND 3 FOLLOWING) AS'max_bmi_current_p3'

    FROM insur

    '''
)

,PatientID,max_bmi_unbounded_p3,max_bmi_current_p3
0,3,34.4,34.4
1,4,37.3,37.3
2,5,37.3,37.3
3,6,37.3,37.3
4,7,37.3,37.3
...,...,...,...
1335,1047,53.1,53.1
1336,1105,53.1,53.1
1337,1124,53.1,53.1
1338,1225,53.1,53.1


#Q11: Problem 11: For each patient, find the rolling average of the last 2 claims.

In [54]:
pysqldf('''
    SELECT PatientID,
    AVG(claim) OVER(ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS 'roll_avg_last2'
    FROM insur

''')

,PatientID,roll_avg_last2
0,1,1121.870000
1,2,1126.690000
2,3,1129.773333
3,4,1134.616667
4,5,1136.450000
...,...,...
1335,1336,52973.596667
1336,1337,55432.433333
1337,1338,57909.290000
1338,1339,60395.113333


# Q12:Problem 12: Find the first claimed insurance value for male and female patients, within each region order the data by patient age in ascending order, and only include patients who are non-diabetic and have a bmi value between 25 and 30.

In [57]:
pysqldf(
    '''
SELECT *,FIRST_VALUE(claim) OVER(PARTITION BY region,gender ORDER BY age ASC) AS'first_claim'
FROM insur
WHERE diabetic = 'No' AND
BMI BETWEEN 25 AND 30


    '''
)

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,first_claim
0,13,14,32.0,male,27.6,100,No,0,No,None,1252.41,1252.41
1,719,720,27.0,female,27.1,103,No,1,No,northeast,10106.13,10106.13
2,852,853,30.0,female,28.9,94,No,2,No,northeast,12096.65,10106.13
3,1134,1135,36.0,female,29.6,99,No,4,No,northeast,24671.66,10106.13
4,893,894,39.0,female,26.5,94,No,0,No,northeast,12815.44,10106.13
...,...,...,...,...,...,...,...,...,...,...,...,...
206,1119,1120,44.0,male,28.0,93,No,1,Yes,southwest,23568.27,25309.49
207,605,606,45.0,male,26.6,99,No,0,No,southwest,8444.47,25309.49
208,425,426,46.0,male,26.9,90,No,0,No,southwest,5969.72,25309.49
209,558,559,46.0,male,27.4,88,No,2,No,southwest,7726.85,25309.49
